# Client B: DEAD (Drinking Excess Alcohol is Dangerous)

In [ ]:
import os
import getpass
import duckdb as db
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

con = db.connect()

RAW = "liquor_2022_2026.parquet"
CENSUS = "liquor_census.parquet"

## 1. Clean zip codes and build the zip x month sales aggregate

In [ ]:
# clean + agg to one row per zip

zip_month = con.execute(f'''
    WITH cleaned AS (
        SELECT
            SUBSTR(TRIM(store_zip_code), 1, 5) AS zip,
            ordered_on,
            strftime(ordered_on, '%Y-%m') AS year_month,
            TRY_CAST(sales_bottles AS DOUBLE) AS sales_bottles,
            sales_dollars,
            sales_liters,
            store_no,
            category_name,
            TRY_CAST(state_bottle_retail AS DOUBLE) AS state_bottle_retail,
            bottle_volume_ml
        FROM '{RAW}'
        -- keep only rows where the zip is exactly 5 digits; drops blanks,
        -- out-of-state garbage, and partial/corrupted zip strings
        WHERE SUBSTR(TRIM(store_zip_code), 1, 5) SIMILAR TO '[0-9]{{5}}'
    )
    SELECT
        zip, year_month,
        SUM(sales_dollars)               AS total_dollars,
        SUM(sales_bottles)               AS total_bottles,
        SUM(sales_liters)                AS total_liters,
        COUNT(DISTINCT store_no)         AS n_stores,
        COUNT(DISTINCT category_name)    AS n_categories,
        COUNT(*)                         AS n_line_items,
        AVG(state_bottle_retail)         AS avg_bottle_retail,
        AVG(bottle_volume_ml)            AS avg_bottle_volume_ml
    FROM cleaned
    GROUP BY zip, year_month
''').df()

# Pull year / month back out of the "YYYY-MM" string for later use as features
zip_month["year"] = zip_month["year_month"].str[:4].astype(int)
zip_month["month_num"] = zip_month["year_month"].str[5:7].astype(int)

# if theres returns in a month that exceed sales, we get negative net liters or dollars; reset to 0
n_neg_liters = (zip_month["total_liters"] < 0).sum()
n_neg_dollars = (zip_month["total_dollars"] < 0).sum()
print(f"{n_neg_liters} zip-month rows have negative net liters, {n_neg_dollars} negative net dollars "
      f"(returns > sales that month); both clipped to 0.")
zip_month["total_liters"] = zip_month["total_liters"].clip(lower=0)
zip_month["total_dollars"] = zip_month["total_dollars"].clip(lower=0)

print(f"{len(zip_month):,} zip-month rows, {zip_month['zip'].nunique()} distinct zips, "
      f"{zip_month['year_month'].nunique()} months")
zip_month.head()

## 1b. Is this actually spirits-only?

In [ ]:
# looking at distinct category labels
categories_seen = con.execute(f"SELECT DISTINCT category_name FROM '{RAW}' ORDER BY category_name").df()
print(f"{len(categories_seen)} distinct category_name values in the raw data:")
for c in categories_seen["category_name"]:
    print(" ", c)

beer_or_wine = categories_seen["category_name"].str.contains("BEER|WINE|MALT", case=False, na=False)
matches = categories_seen.loc[beer_or_wine, "category_name"].tolist()
if matches:
    print(f"\nFOUND beer/wine/malt-related categories -- the spirits-only assumption above is WRONG: {matches}")
else:
    print("\nNo beer/wine/malt categories found -- confirms this dataset is spirits only. "
          "Everything downstream (including the Super Bowl Sunday coefficient in section 11) "
          "should be read as 'spirits purchases,' not 'alcohol purchases' generally.")

## 2. Category-mix features per zip-month

Dollar share of the top categories (statewide ranking, from `liquor_census.parquet`) plus
an "other" bucket. A **predictor** describing purchase composition, not the outcome.

In [ ]:
# Rank the top 10 categories statewide by total dollar sales.
top_categories = con.execute(f'''
    SELECT 
        category_name, 
        SUM(sales_dollars) AS statewide_dollars
    FROM '{CENSUS}'
    WHERE category_name IS NOT NULL
      AND category_name != ''
    GROUP BY category_name
    ORDER BY statewide_dollars DESC
    LIMIT 10
''').df()["category_name"].tolist()

# Bucket every transaction into one of those top-10 categories, or "OTHER",
# and sum dollars per zip-month-bucket.

cat_long = con.execute(f'''
    SELECT
        SUBSTR(TRIM(store_zip_code), 1, 5) AS zip,
        strftime(ordered_on, '%Y-%m') AS year_month,

        -- Collapse anything outside the top 10 into a single "OTHER" bucket
        CASE 
            WHEN category_name IN ({','.join(f"'{c}'" for c in top_categories)})
            THEN category_name 
            ELSE 'OTHER' 
        END AS category_bucket,

        SUM(sales_dollars) AS category_dollars

    FROM '{RAW}'

    -- keep only well-formed 5-digit zips (matches the filter used to build zip_month)
    WHERE SUBSTR(TRIM(store_zip_code), 1, 5) SIMILAR TO '[0-9]{{5}}'

    GROUP BY zip, year_month, category_bucket
''').df()

# Pivot long -> wide: one row per zip-month, one column per category bucket
cat_wide = cat_long.pivot_table(
    index=["zip", "year_month"],
    columns="category_bucket",
    values="category_dollars",
    aggfunc="sum",
    fill_value=0
).reset_index()

# Rename pivoted columns to "share_<category>" (values are still raw dollars
# at this point -- converted to shares a few lines down)
cat_wide.columns = (
    ["zip", "year_month"] +
    [f"share_{c.lower().replace(' ', '_')}" for c in cat_wide.columns[2:]]
)

# Keep track of which columns are the category-share features, for use
# later when building the model's feature list
share_cols = [
    c for c in cat_wide.columns
    if c.startswith("share_")
]

# Attach the category breakdown onto the main zip-month table.
# Both cat_wide and zip_month now come from RAW with the same zip/year_month
# grain, so this merge lines up correctly.
zip_month = zip_month.merge(
    cat_wide,
    on=["zip", "year_month"],
    how="left"
)

# Convert each bucket's raw dollars into a share of that zip-month's total
# spend (dollars / total_dollars). Guard against divide-by-zero with
# replace(0, np.nan), then fill any resulting NaN back to 0 (zip-months
# with $0 total spend get 0% share in every bucket, not NaN).
zip_month[share_cols] = (
    zip_month[share_cols]
    .div(
        zip_month["total_dollars"].replace(0, np.nan),
        axis=0
    )
    .fillna(0)
)

print("category buckets:", share_cols)

# Quick sanity check of the result
zip_month[
    ["zip", "year_month"] + share_cols
].head()

## 3. Price-premium feature (vs. statewide, from `liquor_census.parquet`)

In [ ]:
# How much pricier (or cheaper) is the average bottle in this zip-month than
# the statewide average bottle that same month? Positive = premium pricing.
statewide_price = con.execute(f'''
    SELECT
        strftime(ordered_on, '%Y-%m') AS year_month,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE)) AS statewide_avg_bottle_price
    FROM '{CENSUS}'
    WHERE ordered_on IS NOT NULL
      AND TRY_CAST(state_bottle_retail AS DOUBLE) IS NOT NULL
    GROUP BY strftime(ordered_on, '%Y-%m')
    ORDER BY year_month
''').df()

# Make sure the join key has a matching dtype on both sides before merging
zip_month["year_month"] = zip_month["year_month"].astype(str)
statewide_price["year_month"] = statewide_price["year_month"].astype(str)

zip_month = zip_month.merge(statewide_price, on="year_month", how="left")

zip_month["price_premium_vs_state"] = (
    zip_month["avg_bottle_retail"] / zip_month["statewide_avg_bottle_price"] - 1
)
zip_month["price_premium_vs_state"] = zip_month["price_premium_vs_state"].fillna(0)

zip_month[["zip", "year_month", "avg_bottle_retail", "statewide_avg_bottle_price",
           "price_premium_vs_state"]].head()

## 4. Calendar events (same importer as `caseb_calendar_events.ipynb`, month grain)

Day-count features rather than one dummy per event: every fixed event maps to exactly one
calendar month every year, so an event dummy and a month-of-year dummy carry the same
information at this grain (see `liquor_census_build_data.ipynb`, section 4).

In [ ]:
# Small date-math helpers for events defined as "the Nth weekday of a month" 
def nth_weekday(year, month, weekday, n):
    # e.g. nth_weekday(2024, 9, 0, 1) = the 1st Monday of Sept 2024 (used for Labor Day).
    first = pd.Timestamp(year=year, month=month, day=1)
    return first + pd.Timedelta(days=(weekday - first.weekday()) % 7 + 7 * (n - 1))

def last_weekday(year, month, weekday):
    # e.g. last_weekday(2024, 5, 0) = the last Monday of May 2024 (used for Memorial Day).
    month_end = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    return month_end - pd.Timedelta(days=(month_end.weekday() - weekday) % 7)

# events: {date -> [event names on that date]}. Built up by add_event() below
# so that a date with multiple events (e.g. Dec 24 + Christmas Eve) keeps both.
events = {}
def add_event(date, name):
    events.setdefault(pd.Timestamp(date), []).append(name)

# Fixed dates that move year to year and can't be computed with a simple rule
SUPER_BOWL_SUNDAY = {2022: "2022-02-13", 2023: "2023-02-12", 2024: "2024-02-11",
                      2025: "2025-02-09", 2026: "2026-02-08"}
IOWA_STATE_FAIR = {2022: ("2022-08-11", "2022-08-21"), 2023: ("2023-08-10", "2023-08-20"),
                    2024: ("2024-08-08", "2024-08-18"), 2025: ("2025-08-07", "2025-08-17"),
                    2026: ("2026-08-13", "2026-08-23")}
# Home-game schedules for the two big in-state college football programs --
# these are what section 12's "where" analysis checks for a local (not
# statewide) purchasing bump in Iowa City / Ames.
HAWKEYES_HOME = {
    2022: ["2022-09-03", "2022-09-10", "2022-09-17", "2022-10-01", "2022-10-29", "2022-11-12", "2022-11-25"],
    2023: ["2023-09-02", "2023-09-16", "2023-09-30", "2023-10-07", "2023-10-21", "2023-11-11", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-07", "2024-09-14", "2024-10-12", "2024-10-26", "2024-11-02", "2024-11-29"],
    2025: ["2025-08-30", "2025-09-13", "2025-09-27", "2025-10-18", "2025-10-25", "2025-11-08", "2025-11-22"],
    2026: ["2026-09-05"],
}
CYCLONES_HOME = {
    2022: ["2022-09-03", "2022-09-17", "2022-09-24", "2022-10-08", "2022-10-29", "2022-11-05", "2022-11-19"],
    2023: ["2023-09-02", "2023-09-09", "2023-09-23", "2023-10-07", "2023-11-04", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-21", "2024-10-05", "2024-10-19", "2024-11-02", "2024-11-09", "2024-11-16"],
    2025: ["2025-08-30", "2025-09-06", "2025-09-27", "2025-10-25", "2025-11-01", "2025-11-22"],
    2026: ["2026-09-05", "2026-09-12"],
}

for year in range(2022, 2027):
    add_event(f"{year}-01-01", "New Year's Day")
    add_event(f"{year}-12-31", "New Year's Eve")
    add_event(SUPER_BOWL_SUNDAY[year], "Super Bowl Sunday")
    add_event(f"{year}-03-17", "St. Patrick's Day")
    add_event(f"{year}-05-05", "Cinco de Mayo")
    add_event(last_weekday(year, 5, 0), "Memorial Day")
    add_event(f"{year}-07-04", "July 4th")
    add_event(nth_weekday(year, 9, 0, 1), "Labor Day")
    add_event(f"{year}-10-31", "Halloween")
    thanksgiving = nth_weekday(year, 11, 3, 4)
    add_event(thanksgiving, "Thanksgiving")
    add_event(thanksgiving - pd.Timedelta(days=1), "Thanksgiving")  # the eve counts too
    add_event(f"{year}-12-24", "Christmas")
    add_event(f"{year}-12-25", "Christmas")
    fair_start, fair_end = IOWA_STATE_FAIR[year]
    for day in pd.date_range(fair_start, fair_end):
        add_event(day, "Iowa State Fair")
for date in [d for season in HAWKEYES_HOME.values() for d in season]:
    add_event(date, "Hawkeyes Home Game")
for date in [d for season in CYCLONES_HOME.values() for d in season]:
    add_event(date, "Cyclones Home Game")

events_df = pd.DataFrame([(d, n) for d, n in sorted(events.items())], columns=["date", "events"])
events_df["year_month"] = events_df["date"].dt.strftime("%Y-%m")

EVENT_TYPES = ["New Year's Day", "New Year's Eve", "Super Bowl Sunday", "St. Patrick's Day",
               "Cinco de Mayo", "Memorial Day", "July 4th", "Labor Day", "Halloween",
               "Thanksgiving", "Christmas", "Iowa State Fair", "Hawkeyes Home Game", "Cyclones Home Game"]

def slugify(name):
    return name.lower().replace(" ", "_").replace(".", "").replace("'", "")

# For each month, count how many days that month contain each event type
# (mostly 0 or 1, except multi-day events like the State Fair or a month
# with 2 home games).
month_events = pd.DataFrame({"year_month": sorted(zip_month["year_month"].unique())})
for event_type in EVENT_TYPES:
    mask = events_df["events"].apply(lambda lst: event_type in lst)
    day_counts = events_df.loc[mask].groupby("year_month").size()
    month_events[f"{slugify(event_type)}_days"] = month_events["year_month"].map(day_counts).fillna(0).astype(int)

event_day_cols = [c for c in month_events.columns if c.endswith("_days")]
month_events["n_event_types_in_month"] = (month_events[event_day_cols] > 0).sum(axis=1)

zip_month = zip_month.merge(month_events, on="year_month", how="left")
print(zip_month.shape)
zip_month.head()

## 5. Zip population from the Census API (ACS 5-year, ZCTA)

`store_zip_code` is a USPS ZIP; population is published by **ZCTA**, a Census-drawn
approximation of a ZIP's boundary. Most line up closely, but not always (checked below,
not assumed).

Looking at demographic data via the 2024 American Community Survey (Census)

In [ ]:

zcta_pop = con.execute("""
    SELECT DISTINCT
        CAST(store_zip_code AS VARCHAR) AS zip,
        total_population AS population
    FROM read_parquet('liquor_census.parquet')
    WHERE store_zip_code IS NOT NULL
      AND total_population IS NOT NULL
""").df()

# Clean ZIP codes
zcta_pop["zip"] = (
    zcta_pop["zip"]
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

zcta_pop["population"] = pd.to_numeric(
    zcta_pop["population"],
    errors="coerce"
)

print(f"{len(zcta_pop):,} ZIP codes with population data")

# Same merge as before
zip_month = zip_month.merge(zcta_pop, on="zip", how="left")

zips_in_data = zip_month["zip"].nunique()
zips_missing = zip_month.loc[
    zip_month["population"].isna(), "zip"
].nunique()

print(
    f"{zips_missing} of {zips_in_data} zips in the sales data "
    f"have no matching population"
)

In [ ]:
# ACS 5-year estimates for very small ZCTAs carry large margins of error, and a per-capita
# rate divided by a tiny population is dominated by noise rather than signal. Floor at 100
# residents.
POP_FLOOR = 100
before = len(zip_month)
zip_month = zip_month[zip_month["population"].notna()].copy()
n_dropped_no_pop = before - len(zip_month)
zip_month = zip_month[zip_month["population"] >= POP_FLOOR].copy()
n_dropped_small = before - n_dropped_no_pop - len(zip_month)
print(f"dropped {n_dropped_no_pop:,} rows (no ZCTA population match), "
      f"{n_dropped_small:,} more rows (ZCTA population under {POP_FLOOR})")
print(f"{len(zip_month):,} zip-month rows remain, {zip_month['zip'].nunique()} zips")

# The modeling target: liters per resident, and its log1p transform
zip_month["liters_per_capita"] = zip_month["total_liters"] / zip_month["population"]
zip_month["log_percapita"] = np.log1p(zip_month["liters_per_capita"])
zip_month[["zip", "year_month", "total_liters", "population", "liters_per_capita"]].head()

## 6. Data limitations & ethical considerations — read before the results below

**Scope of the data (see section 1b):** if no beer/wine categories turned up, everything
below is spirits-only. That's the single biggest interpretive caveat in this notebook —
it likely explains why Super Bowl Sunday shows almost no effect in section 11 despite
being a famously beer-heavy occasion, and it means no claim here should be generalized to
"alcohol purchases" without that qualifier.

**Still missing:**

- **Alcohol content (ABV/proof).** `liters_per_capita` is liquid volume per resident, not
  ethanol per resident.
- **Income and ethnicity by zip.** Not in `liquor_2022_2026.parquet` used for Part 1
  (Part 2 below does bring in income/unemployment/poverty/age from the census file, but
  not ethnicity).
- **Sales ≠ consumption.** Wholesale distribution timing, not drinking-day timing.

**Caveats that come with population:**

- **ZCTA ≠ ZIP**, and small-area ACS estimates carry real uncertainty even after the
  ≥100-resident floor in section 5.

**A structural caveat about the statewide model (see sections 11-12):** pooling ~900 zip
codes into one regression can only detect effects that are broadly similar everywhere. An
effect that's large in two college towns and absent elsewhere gets averaged toward zero,
not correctly estimated as "zero, except locally." Section 12 checks this directly for
Iowa City and Ames rather than trusting the pooled coefficient at face value.

**Ethical and legal considerations for how DEAD uses this:**

- **Ecological fallacy.** A zip-month correlation describes an area-level pattern, not
  individual drinking behavior; purchases in a zip include visitors and commuters.
- **Demographic data (Part 2) needs real safeguards** — aggregated only, paired with
  per-capita context, and used to guide voluntary resources, never to target or penalize
  a community.
- **Recommended use of this notebook's output:** section 11 as a *timing* signal
  (statewide, spirits-specific), section 12 as a *where* signal for two specific named
  areas with an actual, checked local effect, and Part 2 as descriptive context on who
  lives in the highest-selling areas — not as a ranking to name-and-shame zip codes (see
  section 14).

## 7. Train / dev / test split (dev + test from 2026)

Train is every month before 2026; dev is Jan-Apr 2026; test is May-Aug 2026. The zip's own
baseline average (`zip_avg_log_percapita`) is computed **from train only** and merged
onto dev/test to avoid leakage.

In [ ]:
zip_month["t"] = (zip_month["year"] - zip_month["year"].min()) * 12 + zip_month["month_num"]
zip_month["t"] = zip_month["t"] - zip_month["t"].min()

all_months = sorted(zip_month["year_month"].unique())
months_2026 = [m for m in all_months if m.startswith("2026")]
train_months = set(m for m in all_months if not m.startswith("2026"))
split_point = len(months_2026) // 2
dev_months = set(months_2026[:split_point])
test_months = set(months_2026[split_point:])

train = zip_month[zip_month["year_month"].isin(train_months)].copy()
dev = zip_month[zip_month["year_month"].isin(dev_months)].copy()
test = zip_month[zip_month["year_month"].isin(test_months)].copy()

# Each zip's own average log-per-capita rate, computed ONLY from the training
# period, then broadcast onto train/dev/test. Computing this from the full
# dataset (including dev/test) would leak future information into a feature
# used to predict those same rows.
zip_baseline = train.groupby("zip")["log_percapita"].mean().rename("zip_avg_log_percapita")
global_baseline = train["log_percapita"].mean()
for d in (train, dev, test):
    d["zip_avg_log_percapita"] = d["zip"].map(zip_baseline).fillna(global_baseline)

feature_cols = (
    ["t", "n_stores", "n_categories", "n_line_items", "avg_bottle_volume_ml",
     "price_premium_vs_state", "zip_avg_log_percapita"]
    + share_cols
    + event_day_cols
    + ["n_event_types_in_month"]
)

X_train, y_train = train[feature_cols].fillna(0), train["log_percapita"]
X_dev, y_dev = dev[feature_cols].fillna(0), dev["log_percapita"]
X_test, y_test = test[feature_cols].fillna(0), test["log_percapita"]

print(f"train: {len(train):,} rows ({min(train_months)}..{max(train_months)})")
print(f"dev:   {len(dev):,} rows, 2026 months {sorted(dev_months)}")
print(f"test:  {len(test):,} rows, 2026 months {sorted(test_months)}")
print(f"{len(feature_cols)} features:", feature_cols)

In [ ]:
print("population" in zip_month.columns, "log_percapita" in zip_month.columns)

## 8. Train models

A zip-average baseline plus three linear models, all predicting
`log1p(liters_per_capita)`, used for absolute-level accuracy (sections 8-10). Section 11
refits a separate model for driver interpretation — see that section for why.

In [ ]:
def evaluate(model, X, y, label):
    # Score log-scale and back-transformed (liters/capita) error metrics for one model.
    pred = model.predict(X)
    return {
        "model": label,
        "RMSE_log": mean_squared_error(y, pred) ** 0.5,
        "MAE_log": mean_absolute_error(y, pred),
        "R2_log": r2_score(y, pred),
        # np.expm1 undoes the log1p transform to put error back in real units
        "RMSE_liters_percapita": mean_squared_error(np.expm1(y), np.expm1(pred)) ** 0.5,
        "MAE_liters_percapita": mean_absolute_error(np.expm1(y), np.expm1(pred)),
    }

class ZipBaseline:
    # Trivial baseline: 'this zip-month will look like that zip's historical average.'
    # Any model we keep needs to beat this to be worth the added complexity.
    def fit(self, X, y): return self
    def predict(self, X): return X["zip_avg_log_percapita"].values

models = {
    "Zip-average baseline": ZipBaseline().fit(X_train, y_train),
    "LinearRegression": LinearRegression().fit(X_train, y_train),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0).fit(X_train, y_train),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=10000).fit(X_train, y_train),
}

dev_results = [evaluate(m, X_dev, y_dev, label) | {"split": "dev"} for label, m in models.items()]
results_df = pd.DataFrame(dev_results).set_index(["model", "split"]).round(4)
results_df

## 9. Test-set evaluation (touched once)

In [ ]:
# The test set is only scored here, once, after all model/feature choices
# above were finalized using dev -- keeps this an honest out-of-sample check.
test_results = [evaluate(m, X_test, y_test, label) | {"split": "test"} for label, m in models.items()]
test_results_df = pd.DataFrame(test_results).set_index(["model", "split"]).round(4)
test_results_df

## 10. Within-zip R²

Subtracting each zip's train-period average from both actual and predicted values
isolates month-to-month movement within a zip. This is a diagnostic on the section-8
models; section 11 fits a dedicated model on this same idea for interpretation.

In [ ]:
def within_zip_r2(df, pred):
    resid_actual = df["log_percapita"] - df["zip_avg_log_percapita"]
    resid_pred = pred - df["zip_avg_log_percapita"]
    return r2_score(resid_actual, resid_pred)

within_rows = [{
    "model": label,
    "within_zip_R2_dev": within_zip_r2(dev, m.predict(X_dev)),
    "within_zip_R2_test": within_zip_r2(test, m.predict(X_test)),
} for label, m in models.items()]
pd.DataFrame(within_rows).set_index("model").round(4)

## 11. Statewide drivers ("when"): a dedicated model, without the zip-baseline bar

In [ ]:
driver_feature_cols = [c for c in feature_cols if c != "zip_avg_log_percapita"]

train_resid_y = y_train - X_train["zip_avg_log_percapita"]
dev_resid_y = y_dev - X_dev["zip_avg_log_percapita"]
test_resid_y = y_test - X_test["zip_avg_log_percapita"]

driver_model = Ridge(alpha=1.0).fit(train[driver_feature_cols].fillna(0), train_resid_y)

print("driver model R\u00b2 on within-zip movement:")
print("  dev: ", r2_score(dev_resid_y, driver_model.predict(dev[driver_feature_cols].fillna(0))))
print("  test:", r2_score(test_resid_y, driver_model.predict(test[driver_feature_cols].fillna(0))))

coef_table = pd.DataFrame({
    "coefficient (log scale)": driver_model.coef_,
    "approx. % change in liters/capita per unit": (np.exp(driver_model.coef_) - 1) * 100,
}, index=driver_feature_cols).sort_values("approx. % change in liters/capita per unit", key=abs, ascending=False)
coef_table.round(3)

In [ ]:
# Horizontal bar chart of the driver-model coefficients, red = negative
# (associated with lower per-capita purchasing), blue = positive.
plot_data = coef_table.sort_values("coefficient (log scale)")
fig, ax = plt.subplots(figsize=(8, max(4, len(plot_data) * 0.3)))
colors = ["tab:red" if v < 0 else "tab:blue" for v in plot_data["coefficient (log scale)"]]
ax.barh(plot_data.index, plot_data["coefficient (log scale)"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Ridge coefficient (log scale, within-zip movement)")
ax.set_title("Statewide 'when' drivers -- within-zip deviation from normal")
plt.tight_layout()
plt.show()

## 12. Iowa City / Ames ("where"): does pooling wash out a local effect?

Exploratory, not validated the same way the statewide model was — roughly a dozen zips
across ~56 months is too little data for a held-out split, so this is fit on everything
available and read directionally, not as a tuned predictive model. The zip lists below are
the standard residential/campus ZIPs for each city (approximate by nature of ZIP
boundaries) and are checked against what's actually present in the data, not assumed.

In [ ]:
IOWA_CITY_ZIPS = {"52240", "52241", "52242", "52243", "52245", "52246"}
AMES_ZIPS = {"50010", "50011", "50012", "50013", "50014"}
college_town_zips = IOWA_CITY_ZIPS | AMES_ZIPS

college_town_data = zip_month[zip_month["zip"].isin(college_town_zips)].copy()
present = sorted(college_town_data["zip"].unique())
print(f"{len(present)} of {len(college_town_zips)} target zips present after the population floor: {present}")
print(f"{len(college_town_data):,} zip-month rows")

if len(college_town_data) > 30:
    # Same within-zip residual idea as section 11, but fit only on these ~12
    # zips, so an effect that's specific to college towns isn't averaged away
    # by the other ~890 zips in the statewide model.
    ct_zip_avg = college_town_data.groupby("zip")["log_percapita"].transform("mean")
    ct_y_resid = college_town_data["log_percapita"] - ct_zip_avg
    ct_X = college_town_data[driver_feature_cols].fillna(0)

    college_model = Ridge(alpha=1.0).fit(ct_X, ct_y_resid)

    compare = pd.DataFrame({
        "statewide (section 11)": pd.Series(driver_model.coef_, index=driver_feature_cols),
        "Iowa City / Ames only": pd.Series(college_model.coef_, index=driver_feature_cols),
    })
    local_event_rows = ["iowa_state_fair_days", "hawkeyes_home_game_days", "cyclones_home_game_days"]
    print("\nlocal-event coefficients, statewide vs. Iowa City/Ames-only:")
    compare.loc[[c for c in local_event_rows if c in compare.index]].round(3)
else:
    print("too few rows in these zips after filtering to fit a meaningful comparison model")

## 13. 2026 actual vs. predicted, by model (statewide, section-8 models)

In [ ]:
plot_df = pd.concat([dev, test]).copy()
for label, m in models.items():
    pred_log = m.predict(plot_df[feature_cols].fillna(0))
    plot_df[f"pred_percapita__{label}"] = np.expm1(pred_log)

months_order = sorted(plot_df["year_month"].unique())
monthly_actual = plot_df.groupby("year_month")["liters_per_capita"].mean().loc[months_order]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(months_order))
ax.plot(x, monthly_actual.values, marker="o", linewidth=2.5, color="black", label="Actual")
for label in models:
    monthly_pred = plot_df.groupby("year_month")[f"pred_percapita__{label}"].mean().loc[months_order]
    ax.plot(x, monthly_pred.values, marker="o", label=label)

ax.axvline(len(dev_months) - 0.5, color="gray", linestyle="--", linewidth=1)
ax.text(len(dev_months) - 0.5, ax.get_ylim()[1], "dev | test", ha="center", va="bottom", fontsize=9, color="gray")
ax.set_xticks(x)
ax.set_xticklabels(months_order, rotation=45)
ax.set_ylabel("Mean liters per capita across zips")
ax.set_title("2026: actual vs. predicted per-capita liters, by model")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter of predicted vs. actual (log scale) for each model, at the
# individual zip-month level -- a tighter cloud around the red y=x line
# means better-calibrated predictions, not just a good average.
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 5), sharex=True, sharey=True)
lims = [plot_df["log_percapita"].min() - 0.2, plot_df["log_percapita"].max() + 0.2]
for ax, label in zip(axes, models):
    pred_log = np.log1p(plot_df[f"pred_percapita__{label}"])
    ax.scatter(plot_df["log_percapita"], pred_log, alpha=0.25, s=12)
    ax.plot(lims, lims, color="red", linewidth=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(label)
    ax.set_xlabel("actual log_percapita")
axes[0].set_ylabel("predicted log_percapita")
plt.suptitle("2026 (dev+test): predicted vs. actual, zip-month level")
plt.tight_layout()
plt.show()

### Appendix A: top ZIPs by per-capita spirits volume

A simple ranking (no modeling) purely as a reference table -- which ZIPs have the highest
average liters-per-capita over the whole period. Read alongside section 14's warning: this
is descriptive, not a "problem areas" list.

In [ ]:
zip_ranking = (
    zip_month.groupby("zip")
    .agg(
        avg_liters_per_capita=("liters_per_capita", "mean"),
        avg_total_liters=("total_liters", "mean"),
        avg_population=("population", "mean"),
        n_months=("year_month", "count"),
    )
    .sort_values("avg_liters_per_capita", ascending=False)
)
zip_ranking.head(20)

In [ ]:
top15 = zip_ranking.head(15).sort_values("avg_liters_per_capita")
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top15.index.astype(str), top15["avg_liters_per_capita"])
ax.set_xlabel("Avg. liters per capita (spirits)")
ax.set_title("Top 15 zips by per-capita spirits volume")
plt.tight_layout()
plt.show()

---
# PART 2: Who lives in the highest-purchasing ZIPs? (supplementary)

## 2.1 Build the zip-month aggregate (with demographics already attached)

In [ ]:
# Re-aggregate from liquor_census.parquet joint witth ACS
DATA = "liquor_census.parquet"

zip_month_2 = con.execute(f'''
    SELECT
        LEFT(TRIM(store_zip_code), 5) AS zip_code,
        strftime(ordered_on, '%Y-%m')  AS year_month,
        YEAR(ordered_on)               AS year,
        MONTH(ordered_on)              AS month_num,

        -- TARGET for this section: raw monthly dollar sales (not per-capita)
        SUM(sales_dollars) AS total_sales,

        -- Other sales/store descriptors
        SUM(sales_liters)                                AS total_liters,
        SUM(TRY_CAST(sales_bottles AS DOUBLE))            AS total_bottles,
        COUNT(DISTINCT store_no)                          AS n_stores,
        COUNT(DISTINCT category_name)                     AS n_categories,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE))      AS avg_bottle_retail,

        -- Demographics (constant per zip-month, so MAX just picks the one value)
        MAX(total_population)          AS total_population,
        MAX(median_household_income)   AS median_household_income,
        MAX(per_capita_income)         AS per_capita_income,
        MAX(unemployment_rate)         AS unemployment_rate,
        MAX(poverty_rate)              AS poverty_rate,
        MAX(median_age)                AS median_age,
        MAX(pct_age_20_34)             AS pct_age_20_34
    FROM '{DATA}'
    WHERE store_zip_code IS NOT NULL
      AND ordered_on IS NOT NULL
      AND sales_dollars IS NOT NULL
    GROUP BY zip_code, year_month, year, month_num
    ORDER BY year, month_num, zip_code
''').df()

zip_month_2.head()

In [ ]:
# Bring in the same calendar-event day-counts built in Part 1, section 4
# (already-built monthly calendar table, kept as its own file so both
# analyses can share it without recomputing).
calendar_monthly = pd.read_parquet("calendar_monthly.parquet")

zip_month_2 = zip_month_2.merge(
    calendar_monthly.drop(columns="date"),
    on=["year", "month_num"],
    how="left",
)

# A single, reusable ranking table: average monthly sales, per-capita sales,
# and population per zip. Built once here and reused by every cell below
# instead of being recomputed with slightly different columns each time.
zip_ranking_2 = (
    zip_month_2.assign(sales_per_capita=lambda d: d["total_sales"] / d["total_population"])
    .groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        total_sales=("total_sales", "sum"),
        avg_population=("total_population", "mean"),
        n_months=("year_month", "count"),
    )
    .sort_values("avg_monthly_sales", ascending=False)
)
zip_ranking_2.head(20)

## 2.2 Look into the top 15 ZIP codes
Who lives here (population, income, unemployment)? Where are these ZIPs located, and do
they share any characteristics? For reference, here's what the top-selling ZIPs
correspond to as cities:

| ZIP Code  | City            |
| --------- | --------------- |
| **50314** | Des Moines      |
| **50320** | Des Moines      |
| **50266** | West Des Moines |
| **51501** | Council Bluffs  |
| **52240** | Iowa City       |
| **52807** | Davenport       |
| **50010** | Ames            |
| **52402** | Cedar Rapids    |
| **52241** | Coralville      |
| **50021** | Ankeny          |
| **50613** | Cedar Falls     |
| **52742** | DeWitt          |
| **50401** | Mason City      |
| **52404** | Cedar Rapids    |
| **50311** | Des Moines      |
| **50702** | Waterloo        |
| **52001** | Dubuque         |
| **50322** | Urbandale       |
| **51106** | Sioux City      |
| **52722** | Bettendorf      |

In [ ]:
top15_zips = zip_ranking_2.head(15).index.astype(str).tolist()

top15 = zip_ranking_2.head(15).reset_index()
top15["zip_code"] = top15["zip_code"].astype(str).str.replace(".0", "", regex=False).str.zfill(5)

fig, ax = plt.subplots(figsize=(7, 6))
plot_top15 = top15.sort_values("avg_monthly_sales")
ax.barh(plot_top15["zip_code"], plot_top15["avg_monthly_sales"])
ax.set_xlabel("Average Monthly Alcohol Sales ($)")
ax.set_ylabel("ZIP Code")
ax.set_title("Top 15 ZIP Codes by Average Monthly Alcohol Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Geographical map of Iowa top 15 ZIPs
import plotly.graph_objects as go

url = "https://raw.githubusercontent.com/OpenDataDE/State-zip-code-GeoJSON/master/ia_iowa_zip_codes_geo.min.json"
iowa_geojson = requests.get(url).json()
all_zips = [feature["properties"]["ZCTA5CE10"] for feature in iowa_geojson["features"]]

fig = go.Figure()

# Base layer: every Iowa ZIP boundary, uncolored, just for outline context
fig.add_trace(go.Choropleth(
    geojson=iowa_geojson, locations=all_zips, featureidkey="properties.ZCTA5CE10",
    z=[0] * len(all_zips), colorscale=[[0, "white"], [1, "white"]], showscale=False,
    marker_line_color="gray", marker_line_width=0.6, hoverinfo="skip",
))

# Highlight layer: only the top-15 zips, colored by average monthly sales
fig.add_trace(go.Choropleth(
    geojson=iowa_geojson, locations=top15["zip_code"], featureidkey="properties.ZCTA5CE10",
    z=top15["avg_monthly_sales"], colorscale="Reds",
    marker_line_color="black", marker_line_width=1.2,
    colorbar_title="Avg. Monthly<br>Sales ($)",
    customdata=top15[["avg_monthly_sales", "total_sales", "avg_population"]],
    hovertemplate=(
        "<b>ZIP %{location}</b><br>"
        "Avg. Monthly Sales: $%{customdata[0]:,.0f}<br>"
        "Total Sales: $%{customdata[1]:,.0f}<br>"
        "Avg. Population: %{customdata[2]:,.0f}"
        "<extra></extra>"
    ),
))

fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")
fig.update_layout(
    title="Top 15 Iowa ZIP Codes by Average Monthly Alcohol Sales",
    margin=dict(l=0, r=0, t=50, b=0), height=650,
)
fig.show()

Who lives in these ZIPs? Are they large-population areas? Higher/lower income? Younger? Higher poverty/unemployment?

In [ ]:
# character analysis of the top 15 zips: demographics, income, age, unemployment, poverty, etc.

top15_demo = (
    zip_month_2[zip_month_2["zip_code"].astype(str).isin(top15_zips)]
    .groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        population=("total_population", "mean"),
        median_income=("median_household_income", "mean"),
        per_capita_income=("per_capita_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
        n_stores=("n_stores", "mean"),
    )
    .sort_values("avg_monthly_sales", ascending=False)
)
top15_demo

Top-15 vs. all-other-ZIPs comparison, on the same demographic measures:

In [ ]:
zip_month_2["group"] = np.where(
    zip_month_2["zip_code"].astype(str).isin(top15_zips), "Top 15 ZIPs", "Other ZIPs"
)

comparison = (
    zip_month_2.groupby("group")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_population=("total_population", "mean"),
        avg_median_income=("median_household_income", "mean"),
        avg_unemployment=("unemployment_rate", "mean"),
        avg_poverty=("poverty_rate", "mean"),
        avg_median_age=("median_age", "mean"),
        avg_pct_age_20_34=("pct_age_20_34", "mean"),
        avg_n_stores=("n_stores", "mean"),
    )
    .round(2)
)
comparison

## 2.3 Trends over time in the top-15 ZIPs

In [ ]:
# Year-by-year average monthly sales for each of the top-15 zips (one line
# per zip, one panel per year, so within-year seasonality and across-year
# trend can both be read without overplotting a single crowded chart).
top15_data = zip_month_2[zip_month_2["zip_code"].astype(str).isin(top15_zips)].copy()

monthly_sales = (
    top15_data.groupby(["year", "month_num"])
    .agg(avg_sales=("total_sales", "mean"))
    .reset_index()
)

month_names_list = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for year in sorted(zip_month_2["year"].unique()):
    data = monthly_sales[monthly_sales["year"] == year].sort_values("month_num")
    plt.figure(figsize=(9, 4))
    plt.plot(data["month_num"], data["avg_sales"], marker="o")
    plt.xticks(range(1, 13), month_names_list)
    plt.xlabel("Month")
    plt.ylabel("Average Monthly Alcohol Sales ($)")
    title_suffix = "(YTD)" if year == zip_month_2["year"].max() else ""
    plt.title(f"Average Alcohol Sales in Top 15 ZIP Codes - {year} {title_suffix}".strip())
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Same top-15 zips, but one line per zip across all years, to see whether a
# given zip's rank is stable year over year or whether it moves around
top15_yearly = (
    top15_data[top15_data["year"] != top15_data["year"].max()]  
    .groupby(["zip_code", "year"])
    .agg(total_sales=("total_sales", "sum"), avg_monthly_sales=("total_sales", "mean"))
    .reset_index()
)

plt.figure(figsize=(9, 5))
for zip_code in top15_zips:
    temp = top15_yearly[top15_yearly["zip_code"].astype(str) == str(zip_code)]
    plt.plot(temp["year"], temp["avg_monthly_sales"], marker="o", label=zip_code)
plt.xlabel("Year")
plt.ylabel("Average Monthly Alcohol Sales ($)")
plt.title("Alcohol Sales in Top 15 ZIP Codes, by Year")
plt.xticks(sorted(top15_yearly["year"].unique()))
plt.legend(title="ZIP", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 2.4 Do calendar events actually lift sales in these top ZIPs?

In [ ]:
# For each event flag available in the merged calendar table, compare average
# sales on event months/days vs. non-event ones, within the top-15 zips only.
events_to_check = [
    "new_years_day", "new_years_eve", "super_bowl_sunday", "st_patricks_day",
    "cinco_de_mayo", "memorial_day", "july_4th", "labor_day", "halloween",
    "thanksgiving", "christmas", "iowa_state_fair", "hawkeyes_home_game", "cyclones_home_game",
]

event_results = []
for event in events_to_check:
    if event in top15_data.columns:
        event_sales = top15_data.loc[top15_data[event] == 1, "total_sales"].mean()
        non_event_sales = top15_data.loc[top15_data[event] == 0, "total_sales"].mean()
        event_results.append({
            "event": event,
            "event_avg_sales": event_sales,
            "non_event_avg_sales": non_event_sales,
            "difference": event_sales - non_event_sales,
        })

event_comparison = pd.DataFrame(event_results)
event_comparison.sort_values("difference", ascending=False)

## 2.5 Population vs. sales, and seasonality across all top-15 ZIPs

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(zip_month_2["total_population"], zip_month_2["total_sales"], alpha=0.4)
plt.xlabel("Population")
plt.ylabel("Monthly Alcohol Sales ($)")
plt.title("Population vs. Monthly Alcohol Sales")
plt.tight_layout()
plt.show()

In [ ]:
# average monthly sales across all years, by month, for the top 15 zips
top15_monthly = (
    top15_data.groupby("month_num").agg(avg_sales=("total_sales", "mean")).reset_index()
)

plt.figure(figsize=(9, 5))
plt.plot(top15_monthly["month_num"], top15_monthly["avg_sales"], marker="o")
plt.xticks(range(1, 13), month_names_list)
plt.xlabel("Month")
plt.ylabel("Average Monthly Alcohol Sales ($)")
plt.title("Average Alcohol Sales in Top 15 Iowa ZIP Codes by Month")
plt.tight_layout()
plt.show()

## 2.6 A "campaign priority" score: which ZIPs are elevated on multiple signals?

ZIP codes flagged based on the following criteria:
- what we mark as priority

- **Volume** — is average monthly sales high in absolute terms?
- **Per capita** — is purchasing high relative to the ZIP's own population?
- **Higher than expected** — does the ZIP sell more than a simple model (population,
  store count, demographics, seasonality) would predict?

A ZIP flagged on 2+ of these is less likely to be a fluke of any single measure.

In [ ]:
# Step 1: fit candidate models to find which set of predictors explains raw monthly sales best, validated with a simple expanding time-based split
# (train on earlier years, validate on the next year) rather than a random split, since random-splitting time series data leaks future information into training.
df2 = zip_month_2[zip_month_2["year"].isin([2022, 2023, 2024, 2025])].copy()

month_dummies = pd.get_dummies(df2["month_num"], prefix="month", drop_first=True, dtype=int)
df2 = pd.concat([df2, month_dummies], axis=1)
month_features = list(month_dummies.columns)

target = "total_sales"

market_model = ["total_population", "n_stores"]
demographic_model = market_model + [
    "median_household_income", "unemployment_rate", "poverty_rate", "median_age", "pct_age_20_34",
]
season_model = demographic_model + month_features
event_model = season_model + ["iowa_state_fair", "hawkeyes_home_game", "cyclones_home_game"]

candidate_models = {
    "Market Size": market_model,
    "Demographics": demographic_model,
    "Seasonality": season_model,
    "Seasonality + Events": event_model,
}

# Fold 1: train on 2022, validate on 2023. Fold 2: train on 2022-2023, validate on 2024.
folds = [([2022], 2023), ([2022, 2023], 2024)]

validation_results = []
for model_name, features in candidate_models.items():
    for train_years, val_year in folds:
        train_fold = df2[df2["year"].isin(train_years)].dropna(subset=features + [target])
        val_fold = df2[df2["year"] == val_year].dropna(subset=features + [target])

        model = LinearRegression().fit(train_fold[features], train_fold[target])
        pred = model.predict(val_fold[features])

        validation_results.append({
            "model": model_name,
            "validation_year": val_year,
            "RMSE": np.sqrt(mean_squared_error(val_fold[target], pred)),
            "MAE": mean_absolute_error(val_fold[target], pred),
            "R2": r2_score(val_fold[target], pred),
        })

validation_results = pd.DataFrame(validation_results)
model_comparison = (
    validation_results.groupby("model")
    .agg(avg_RMSE=("RMSE", "mean"), avg_MAE=("MAE", "mean"), avg_R2=("R2", "mean"))
    .sort_values("avg_RMSE")
)
print("================ MODEL SELECTION ================")
print(model_comparison.round(3))

best_model_name = model_comparison.index[0]
best_features = candidate_models[best_model_name]
print("\nSelected model:", best_model_name)
print("Selected features:", best_features)

In [ ]:
# Step 2: refit the selected model on 2022-2024, hold out 2025 as a final
train_final = df2[df2["year"].isin([2022, 2023, 2024])].dropna(subset=best_features + [target]).copy()
test_final = df2[df2["year"] == 2025].dropna(subset=best_features + [target]).copy()

final_model = LinearRegression().fit(train_final[best_features], train_final[target])
test_final["predicted_sales"] = final_model.predict(test_final[best_features])
test_final["residual"] = test_final["total_sales"] - test_final["predicted_sales"]

test_rmse = np.sqrt(mean_squared_error(test_final["total_sales"], test_final["predicted_sales"]))
test_mae = mean_absolute_error(test_final["total_sales"], test_final["predicted_sales"])
test_r2 = r2_score(test_final["total_sales"], test_final["predicted_sales"])

print("================ 2025 HOLD-OUT RESULTS ================")
print(f"RMSE: ${test_rmse:,.2f}")
print(f"MAE:  ${test_mae:,.2f}")
print(f"R\u00b2:   {test_r2:.3f}")

coefficients = pd.DataFrame({"feature": best_features, "coefficient": final_model.coef_})
print("\n================ COEFFICIENTS ================")
print(coefficients.sort_values("coefficient", ascending=False).to_string(index=False))

In [ ]:
# Step 3: build the three purchasing signals across all complete years
# (2022-2025), using the fitted model to score "higher than expected."
campaign = df2.dropna(subset=best_features + [target, "total_population"]).copy()
campaign["predicted_sales"] = final_model.predict(campaign[best_features])
campaign["residual"] = campaign["total_sales"] - campaign["predicted_sales"]
campaign["above_expected"] = (campaign["residual"] > 0).astype(int)
campaign["sales_per_capita"] = campaign["total_sales"] / campaign["total_population"]

zip_priority = (
    campaign.groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        months_above_expected=("above_expected", "sum"),
        months_observed=("above_expected", "count"),
        population=("total_population", "mean"),
    )
    .reset_index()
)
zip_priority["pct_months_above_expected"] = (
    zip_priority["months_above_expected"] / zip_priority["months_observed"] * 100
)

# "High" on each signal = top quartile (75th percentile) of all zips
sales_cutoff = zip_priority["avg_monthly_sales"].quantile(0.75)
percap_cutoff = zip_priority["avg_sales_per_capita"].quantile(0.75)
residual_cutoff = zip_priority["avg_residual"].quantile(0.75)

zip_priority["high_volume"] = (zip_priority["avg_monthly_sales"] >= sales_cutoff).astype(int)
zip_priority["high_per_capita"] = (zip_priority["avg_sales_per_capita"] >= percap_cutoff).astype(int)
zip_priority["high_residual"] = (zip_priority["avg_residual"] >= residual_cutoff).astype(int)
zip_priority["priority_signals"] = (
    zip_priority["high_volume"] + zip_priority["high_per_capita"] + zip_priority["high_residual"]
)

zip_priority = zip_priority.sort_values(
    ["priority_signals", "pct_months_above_expected", "avg_monthly_sales"], ascending=False
)

print("================ WHERE: CAMPAIGN-PRIORITY ZIP CODES ================")
print(
    zip_priority[["zip_code", "avg_monthly_sales", "avg_sales_per_capita",
                  "avg_residual", "pct_months_above_expected", "priority_signals"]]
    .head(20).round(2).to_string(index=False)
)

In [ ]:
# Step 4: among ZIPs flagged on 2+ signals, find which calendar months are recurring high points (the "when" companion to the "where" above).
priority_zips = zip_priority.loc[zip_priority["priority_signals"] >= 2, "zip_code"].astype(str)
priority_data = campaign[campaign["zip_code"].astype(str).isin(priority_zips)].copy()

month_names = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
               7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

month_priority = (
    priority_data.groupby("month_num")
    .agg(
        avg_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        pct_observations_above_expected=("above_expected", "mean"),
    )
    .reset_index()
)
month_priority["pct_observations_above_expected"] *= 100
month_priority["month"] = month_priority["month_num"].map(month_names)
month_priority = month_priority.sort_values("avg_sales_per_capita", ascending=False)

print("================ WHEN: CAMPAIGN-PRIORITY MONTHS ================")
print(
    month_priority[["month", "avg_sales", "avg_sales_per_capita", "avg_residual",
                     "pct_observations_above_expected"]]
    .round(2).to_string(index=False)
)

In [ ]:
# Step 5: combine "where" and "when" -- for priority zips, which
# ZIP x calendar-month combinations 
# looking at the top quartile of zip codes
zip_month_priority = (
    priority_data.groupby(["zip_code", "month_num"])
    .agg(
        avg_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        years_above_expected=("above_expected", "sum"),
        years_observed=("above_expected", "count"),
    )
    .reset_index()
)
zip_month_priority["month"] = zip_month_priority["month_num"].map(month_names)
zip_month_priority["pct_years_above_expected"] = (
    zip_month_priority["years_above_expected"] / zip_month_priority["years_observed"] * 100
)

campaign_targets = zip_month_priority[
    (zip_month_priority["avg_residual"] > 0) & (zip_month_priority["pct_years_above_expected"] >= 75)
].sort_values(["pct_years_above_expected", "avg_sales_per_capita", "avg_sales"], ascending=False)

print("================ WHERE + WHEN: RECURRING ELEVATED ZIP-MONTHS ================")
print(
    campaign_targets[["zip_code", "month", "avg_sales", "avg_sales_per_capita",
                       "avg_residual", "pct_years_above_expected"]]
    .head(30).round(2).to_string(index=False)
)

### Visualizing the priority ZIPs and their monthly pattern

In [ ]:
# looking at whos buying the most
from IPython.display import display
import seaborn as sns

priority_table = (
    zip_priority[["zip_code", "avg_monthly_sales", "avg_sales_per_capita",
                  "avg_residual", "pct_months_above_expected", "priority_signals"]]
    .sort_values(["priority_signals", "avg_sales_per_capita"], ascending=False)
    .head(15)
    .copy()
)

print("TOP CAMPAIGN-PRIORITY ZIP CODES")
display(priority_table.style.format({
    "avg_monthly_sales": "${:,.0f}", "avg_sales_per_capita": "${:,.2f}",
    "avg_residual": "${:,.0f}", "pct_months_above_expected": "{:.1f}%",
}))

plot_data = priority_table.sort_values("avg_sales_per_capita")
plt.figure(figsize=(10, 7))
plt.barh(plot_data["zip_code"].astype(str), plot_data["avg_sales_per_capita"])
plt.xlabel("Average Monthly Sales Per Capita ($)")
plt.ylabel("ZIP Code")
plt.title("Alcohol Purchasing Intensity in Campaign-Priority ZIP Codes")
plt.tight_layout()
plt.show()

In [ ]:
# monthly avg sales in priority zips
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

monthly_table = month_priority.copy()
monthly_table["month"] = pd.Categorical(monthly_table["month"], categories=month_order, ordered=True)
monthly_table = monthly_table.sort_values("month")

print("MONTHLY PATTERNS IN PRIORITY ZIP CODES")
display(monthly_table[["month", "avg_sales", "avg_sales_per_capita", "avg_residual",
                        "pct_observations_above_expected"]].style.format({
    "avg_sales": "${:,.0f}", "avg_sales_per_capita": "${:,.2f}",
    "avg_residual": "${:,.0f}", "pct_observations_above_expected": "{:.1f}%",
}))

plt.figure(figsize=(11, 5))
plt.plot(monthly_table["month"], monthly_table["avg_sales_per_capita"], marker="o")
plt.xlabel("Month")
plt.ylabel("Average Sales Per Capita ($)")
plt.title("When Is Alcohol Purchasing Highest in Priority ZIP Codes?")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# how far above/below each zip's own normal is each month, on average, for the priority zips
# normalized

heatmap_data = priority_data.copy()
heatmap_data["zip_normal"] = heatmap_data.groupby("zip_code")["sales_per_capita"].transform("mean")
heatmap_data["pct_from_zip_normal"] = (heatmap_data["sales_per_capita"] / heatmap_data["zip_normal"] - 1) * 100

zip_month_pattern = (
    heatmap_data.groupby(["zip_code", "month_num"])
    .agg(
        avg_pct_from_normal=("pct_from_zip_normal", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_total_sales=("total_sales", "mean"),
    )
    .reset_index()
)

top_zips_for_heatmap = (
    zip_priority.sort_values(["priority_signals", "avg_monthly_sales"], ascending=False)
    .head(15)["zip_code"]
)

heatmap_plot = (
    zip_month_pattern[zip_month_pattern["zip_code"].isin(top_zips_for_heatmap)]
    .pivot(index="zip_code", columns="month_num", values="avg_pct_from_normal")
    .reindex(columns=range(1, 13))
)
heatmap_plot.columns = month_names_list

plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_plot, annot=True, fmt=".0f", center=0, cmap="RdBu_r",
    cbar_kws={"label": "% Above/Below ZIP's Normal Purchasing"},
)
plt.xlabel("Month")
plt.ylabel("ZIP Code")
plt.title("Where and When Is Alcohol Purchasing Elevated?\nAverage Pattern, 2022-2025")
plt.tight_layout()
plt.show()

In [ ]:
pattern_table = zip_month_pattern[zip_month_pattern["zip_code"].isin(top_zips_for_heatmap)].copy()
pattern_table["month"] = pattern_table["month_num"].map(month_names)
# recurring eleveated periods (useful for figuring out high impact times)
pattern_table = pattern_table[pattern_table["avg_pct_from_normal"] > 0].sort_values(
    "avg_pct_from_normal", ascending=False
)

print("STRONGEST RECURRING WHERE + WHEN PATTERNS")
display(
    pattern_table[["zip_code", "month", "avg_total_sales", "avg_sales_per_capita", "avg_pct_from_normal"]]
    .head(30)
    .style.format({
        "avg_total_sales": "${:,.0f}", "avg_sales_per_capita": "${:,.2f}", "avg_pct_from_normal": "+{:.1f}%",
    })
)

## 2.7 Demographic profile of priority ZIPs vs. everyone else

In [ ]:
zip_context = (
    campaign.groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        pct_months_above_expected=("above_expected", lambda x: x.mean() * 100),
        population=("total_population", "mean"),
        avg_stores=("n_stores", "mean"),
        median_income=("median_household_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
    )
    .reset_index()
)

volume_cut = zip_context["avg_monthly_sales"].quantile(0.75)
percap_cut = zip_context["avg_sales_per_capita"].quantile(0.75)
residual_cut = zip_context["avg_residual"].quantile(0.75)

zip_context["high_volume"] = (zip_context["avg_monthly_sales"] >= volume_cut).astype(int)
zip_context["high_per_capita"] = (zip_context["avg_sales_per_capita"] >= percap_cut).astype(int)
zip_context["higher_than_expected"] = (zip_context["avg_residual"] >= residual_cut).astype(int)
zip_context["purchasing_signals"] = (
    zip_context["high_volume"] + zip_context["high_per_capita"] + zip_context["higher_than_expected"]
)

priority_context = zip_context[zip_context["purchasing_signals"] >= 2].sort_values(
    ["purchasing_signals", "avg_sales_per_capita"], ascending=False
)

print("DEMOGRAPHIC PROFILE OF CAMPAIGN-PRIORITY ZIPs")
display(
    priority_context[["zip_code", "avg_monthly_sales", "avg_sales_per_capita", "purchasing_signals",
                       "population", "median_income", "unemployment_rate", "poverty_rate",
                       "median_age", "pct_age_20_34", "avg_stores"]]
    .head(20)
    .style.format({
        "avg_monthly_sales": "${:,.0f}", "avg_sales_per_capita": "${:,.2f}",
        "population": "{:,.0f}", "median_income": "${:,.0f}",
        "unemployment_rate": "{:.1f}%", "poverty_rate": "{:.1f}%",
        "median_age": "{:.1f}", "pct_age_20_34": "{:.1f}%", "avg_stores": "{:.1f}",
    })
)

In [ ]:
# Priority zips vs. everyone else, side by side, on the same demographic
# measures the aggregate version of the table above.
zip_context["group"] = np.where(
    zip_context["purchasing_signals"] >= 2, "Campaign-priority ZIPs", "Other Iowa ZIPs"
)

group_comparison = (
    zip_context.groupby("group")
    .agg(
        avg_monthly_sales=("avg_monthly_sales", "mean"),
        sales_per_capita=("avg_sales_per_capita", "mean"),
        population=("population", "mean"),
        median_income=("median_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
        avg_stores=("avg_stores", "mean"),
    )
    .round(2)
)
display(group_comparison)

# AI Disclosure

The code in this notebook was generated by AI under direct step-by-step specification of each stage throughout the analysis. We thoroughly reviewed, tested, and corrected all generated code. We are responsible for the analysis, interpretation, and conclusions presented.